# EGH444 Assessment 2 — Group workflow

Task 1 runs **both RGB baselines through one script** (`Task_1/rgb_segmentation.py`) so they share an identical protocol:

| Setting | Value (both models) |
|---|---|
| Splits | `train.txt` shuffled with seed 42 → 636 train / 159 val; `test.txt` → 654 test |
| Labels | raw IDs 1–40 → 0–39; raw 0 (unlabelled) ignored |
| Input size | 120 × 160 |
| Backbone | ImageNet ResNet50 (dilated, output stride 8), only `layer4` fine-tuned, backbone BN stats frozen |
| Optimiser | AdamW, head LR 1e-3, backbone LR 1e-4, poly decay, ≤12 epochs, early stop (patience 4), batch 2 |
| Model selection | best **val** mIoU; test evaluated only on the selected checkpoint |
| Scoring | every model scored on **240×320** label masks (predictions upsampled), val and test |

The only variable is the head: **FCN** (single 3×3 conv) vs **DeepLabV3** (ASPP multi-scale context).

## Setup

In [ ]:
%pip -q install gdown

import json
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

import gdown
import pandas as pd
from IPython.display import display

In [ ]:
# Clone the repo, or update it if it's already here.
# If the repo is ever made private again, save a GitHub token as a Colab secret named GITHUB_TOKEN.
REPO = Path("/content/EGH444-Assessment2")
BRANCH = "task1-unified-baselines"   # switch to "master" once this is merged
try:
    from google.colab import userdata
    URL = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/cal272b/EGH444-Assessment2.git"
except Exception:
    URL = "https://github.com/cal272b/EGH444-Assessment2.git"


def git(*args):
    subprocess.run(["git", *args], check=True)   # raises instead of failing silently


if REPO.exists() and not (REPO / ".git").exists():
    shutil.rmtree(REPO)                            # leftover from a failed clone
if REPO.exists():
    git("-C", REPO, "remote", "set-url", "origin", URL)
    git("-C", REPO, "fetch", "-q", "origin", BRANCH)
    git("-C", REPO, "checkout", "-q", "-B", BRANCH, f"origin/{BRANCH}")
else:
    git("clone", "-q", "-b", BRANCH, URL, REPO)
print("Code at", subprocess.run(["git", "-C", REPO, "log", "-1", "--format=%h %s"],
                                capture_output=True, text=True).stdout)

TASK_1 = REPO / "Task_1"
SCRIPT = TASK_1 / "rgb_segmentation.py"
RUNS = TASK_1 / "runs"
sys.path.insert(0, str(TASK_1))


def run(*args):
    """Run rgb_segmentation.py and stream its output into the cell."""
    cmd = [sys.executable, str(SCRIPT), *map(str, args)]
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as p:
        for line in p.stdout:
            print(line, end="")
    if p.returncode:
        raise RuntimeError(f"Command failed ({p.returncode})")

## Dataset

In [ ]:
DATA_ROOT = Path("/content/datasets/NYUDepthv2")

if not (DATA_ROOT / "train.txt").exists():
    zip_path = "/content/NYUDepthv2.zip"
    gdown.download(id="1nGsWNF7zU0O6RZAG-aQIs4HYnxSwP-mr", output=zip_path)
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall("/content/datasets")
    Path(zip_path).unlink(missing_ok=True)
    DATA_ROOT = next(
        p.parent for p in Path("/content/datasets").rglob("train.txt")
        if (p.parent / "RGB").is_dir() and (p.parent / "Label").is_dir()
    )
print("Dataset:", DATA_ROOT)

## Task 1 — Checkpoints

Each trained model lives at `runs/<run>/<model>/best.pt`, with that run's `splits.json` two levels up
(the script refuses to evaluate a checkpoint on different sample IDs).

In [ ]:
# Back up / restore trained runs on Google Drive so a Colab reset doesn't lose them
USE_DRIVE = True
DRIVE_RUNS = Path("/content/drive/MyDrive/EGH444/runs")

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)   # a stale mount gives "Transport endpoint is not connected"
    DRIVE_RUNS.mkdir(parents=True, exist_ok=True)


def restore(run_name):
    src = DRIVE_RUNS / run_name
    if USE_DRIVE and src.exists() and not (RUNS / run_name).exists():
        try:
            shutil.copytree(src, RUNS / run_name)
            print("Restored", run_name, "from Drive")
        except Exception as e:
            shutil.rmtree(RUNS / run_name, ignore_errors=True)   # don't leave a half-copied run behind
            raise RuntimeError(f"Drive copy failed ({e}). Re-run the Drive mount cell, then this cell.") from None


def backup(run_name):
    if USE_DRIVE:
        shutil.copytree(RUNS / run_name, DRIVE_RUNS / run_name, dirs_exist_ok=True)
        print("Backed up", run_name, "to Drive")

In [ ]:
# DeepLabV3 (Dylan) — existing trained weights (120x160) + the original run's splits/config
DEEPLAB_RUN = "deeplab-full-corrected"
DEEPLAB_CKPT = RUNS / DEEPLAB_RUN / "deeplabv3/best.pt"
DEEPLAB_CKPT.parent.mkdir(parents=True, exist_ok=True)
if not DEEPLAB_CKPT.exists():
    gdown.download(id="1nLs0MFeoaaIBUxXlq7iwrF-pexfdW820", output=str(DEEPLAB_CKPT))
for filename in ["splits.json", "config.json"]:
    shutil.copy2(TASK_1 / "DeepLab_Results" / DEEPLAB_RUN / filename, RUNS / DEEPLAB_RUN / filename)

## All experiments

**One protocol for every number in the report:** same splits (seed 42: 636 train / 159 val / 654 test), same recipe
(table at top), and **every model is scored on 240×320 label masks** (lower-resolution models' predictions are
bilinearly upsampled to 240×320 before argmax). Each run trains only if it isn't already on Drive.

| Task | Label | Change vs its baseline |
|---|---|---|
| 1 | DeepLabV3, FCN | Baselines at 120×160 input (only the head differs) |
| 1 | DeepLabV3 240×320 | Does ASPP's multi-scale context matter once the feature map is larger (30×40)? Compare with FCN 240×320 |
| 2 | A: DeepLabV3 ASPP 3-6-9 | Atrous rates sized to the 15×20 feature map |
| 2 | B: FCN weighted-CE | Median-frequency class weights |
| 2 | C: FCN 240×320 | Input resolution 120×160 → 240×320 |
| 2 | FCN seed 1/2 · FCN 240×320 seed 1/2 | Run-to-run noise (same splits, different training seed) |
| 3 | D: RGB-D mid (+ seeds 1/2) | Depth encoder fused before the head (on FCN 240×320) |
| 3 | E: RGB-D early | Depth as a 4th input channel (on FCN 240×320) |

In [ ]:
EVAL_SIZE = (240, 320)
S240 = ["--size", 240, 320]
SEEDS = lambda s: ["--seed", s, "--split-seed", 42]

# label: (run folder, training flags)  — flags None = pre-trained checkpoint (Dylan's)
EXPERIMENTS = {
    "DeepLabV3":                    ("deeplab-full-corrected", None),
    "FCN":                          ("fcn-baseline", ["--model", "fcn"]),
    "FCN seed 1":                   ("fcn-seed1", ["--model", "fcn", *SEEDS(1)]),
    "FCN seed 2":                   ("fcn-seed2", ["--model", "fcn", *SEEDS(2)]),
    "DeepLabV3 ASPP 3-6-9":         ("deeplab-aspp369", ["--model", "deeplabv3", "--aspp-rates", 3, 6, 9]),
    "FCN weighted-CE":              ("fcn-weightedce", ["--model", "fcn", "--loss", "weighted-ce"]),
    "FCN 240x320":                  ("fcn-240x320", ["--model", "fcn", *S240]),
    "FCN 240x320 seed 1":           ("fcn-240x320-seed1", ["--model", "fcn", *S240, *SEEDS(1)]),
    "FCN 240x320 seed 2":           ("fcn-240x320-seed2", ["--model", "fcn", *S240, *SEEDS(2)]),
    "DeepLabV3 240x320":            ("deeplab-240x320", ["--model", "deeplabv3", *S240]),
    "FCN 240x320 RGB-D mid":        ("fcn-240x320-rgbd-mid", ["--model", "fcn", *S240, "--depth", "mid"]),
    "FCN 240x320 RGB-D mid seed 1": ("fcn-240x320-rgbd-mid-seed1", ["--model", "fcn", *S240, "--depth", "mid", *SEEDS(1)]),
    "FCN 240x320 RGB-D mid seed 2": ("fcn-240x320-rgbd-mid-seed2", ["--model", "fcn", *S240, "--depth", "mid", *SEEDS(2)]),
    "FCN 240x320 RGB-D early":      ("fcn-240x320-rgbd-early", ["--model", "fcn", *S240, "--depth", "early"]),
}


def checkpoint(run_name, flags):
    """Train the run unless it's already here or on Drive; return its best.pt."""
    if flags is None:
        return DEEPLAB_CKPT
    restore(run_name)
    model = flags[flags.index("--model") + 1]
    ckpt = RUNS / run_name / model / "best.pt"
    if not ckpt.exists():
        shutil.rmtree(RUNS / run_name, ignore_errors=True)   # clear a half-finished run
        run("--data-root", DATA_ROOT, "--output-root", RUNS, "--run-name", run_name, *flags)
        backup(run_name)
    return ckpt


def evaluate(label, ckpt):
    """Score a checkpoint on val + test at EVAL_SIZE (cached for this session)."""
    out = RUNS / "eval" / label.lower().replace(" ", "-")
    if not (out / "test" / "metrics.json").exists():
        shutil.rmtree(out, ignore_errors=True)
        run("--data-root", DATA_ROOT, "--evaluate-checkpoint", ckpt, "--eval-size", *EVAL_SIZE,
            "--output-root", out.parent, "--run-name", out.name)
    return out


EVAL = {}
for label, (run_name, flags) in EXPERIMENTS.items():
    print(f"\n=== {label} ===")
    EVAL[label] = evaluate(label, checkpoint(run_name, flags))

In [ ]:
# Master results table — every model, one protocol (240x320 labels). Copy rows from here into the report.
def master_row(label):
    row = {"model": label}
    for split in ["val", "test"]:
        m = json.loads((EVAL[label] / split / "metrics.json").read_text())
        row |= {f"{split}_mIoU": 100 * m["mIoU"], f"{split}_PixAcc": 100 * m["PixelAcc"], f"{split}_Dice": 100 * m["mean_Dice_F1"]}
    row |= {"test_boundary_acc": 100 * m["boundary_PixelAcc"], "test_interior_acc": 100 * m["interior_PixelAcc"],
            "infer_ms": m["inference_ms_per_image"]}
    run_name, flags = EXPERIMENTS[label]
    history = next((RUNS / run_name).glob("*/training.csv"), None)
    if history is not None:
        h = pd.read_csv(history)
        row |= {"sec_per_epoch": h.epoch_seconds.mean(), "best_epoch": int(h.loc[h.val_mIoU.idxmax(), "epoch"]),
                "peak_GPU_MB": h.peak_gpu_MB.max() if "peak_gpu_MB" in h else None}
    return row

MASTER = pd.DataFrame([master_row(k) for k in EVAL]).set_index("model").round(2)
MASTER.to_csv(TASK_1 / "master_results.csv")
display(MASTER)

In [ ]:
# Run-to-run noise and the paired depth effect (3 training seeds each, identical splits)
groups = {"FCN 120x160": ["FCN", "FCN seed 1", "FCN seed 2"],
          "FCN 240x320": ["FCN 240x320", "FCN 240x320 seed 1", "FCN 240x320 seed 2"],
          "FCN 240x320 RGB-D mid": ["FCN 240x320 RGB-D mid", "FCN 240x320 RGB-D mid seed 1", "FCN 240x320 RGB-D mid seed 2"]}
noise = pd.DataFrame({g: MASTER.loc[m, ["val_mIoU", "test_mIoU"]].agg(["mean", "std"]).unstack() for g, m in groups.items()}).T.round(2)
display(noise)

paired = pd.DataFrame({"seed": [42, 1, 2],
                       "RGB test mIoU": MASTER.loc[groups["FCN 240x320"], "test_mIoU"].values,
                       "RGB-D mid test mIoU": MASTER.loc[groups["FCN 240x320 RGB-D mid"], "test_mIoU"].values})
paired["depth gain"] = paired["RGB-D mid test mIoU"] - paired["RGB test mIoU"]
display(paired.round(2))
g = paired["depth gain"]
print(f"Depth gain: mean {g.mean():+.2f} mIoU, std {g.std():.2f}, positive in {(g > 0).sum()}/3 seeds")

In [ ]:
# Learning curves: training loss keeps falling while val mIoU plateaus (overfitting with 636 images)
import matplotlib.pyplot as plt
curves = ["FCN", "FCN 240x320", "FCN 240x320 RGB-D mid", "DeepLabV3 240x320"]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
for label in curves:
    h = pd.read_csv(next((RUNS / EXPERIMENTS[label][0]).glob("*/training.csv")))
    ax1.plot(h.epoch, h.train_loss, marker="o", ms=3, label=label)
    ax2.plot(h.epoch, 100 * h.val_mIoU, marker="o", ms=3, label=label)
ax1.set(xlabel="epoch", ylabel="training loss", title="Training loss")
ax2.set(xlabel="epoch", ylabel="val mIoU (%) at training resolution", title="Validation mIoU")
for a in (ax1, ax2):
    a.grid(alpha=0.3); a.spines[["top", "right"]].set_visible(False)
ax2.legend(frameon=False, fontsize=8)
plt.tight_layout(); fig.savefig(TASK_1 / "learning_curves.png", dpi=150); plt.show()

## Pairwise comparisons (class-wise IoU, qualitative panels)

In [ ]:
import importlib
importlib.invalidate_caches()   # Task_1 may not have existed when sys.path was first scanned
from compare_runs import compare
from failure_analysis import top_confusions, scene_breakdown, boundary_summary, worst_images, show_examples, per_image_delta, confusion_change

COMPARISONS = {
    "task1_baselines":      ("DeepLabV3", "FCN"),
    "task1_baselines_240":  ("DeepLabV3 240x320", "FCN 240x320"),
    "task2_A_aspp":         ("DeepLabV3", "DeepLabV3 ASPP 3-6-9"),
    "task2_B_weightedce":   ("FCN", "FCN weighted-CE"),
    "task2_C_resolution":   ("FCN", "FCN 240x320"),
    "task3_D_mid":          ("FCN 240x320", "FCN 240x320 RGB-D mid"),
    "task3_E_early":        ("FCN 240x320", "FCN 240x320 RGB-D early"),
}
CMP = {}
for name, (base, other) in COMPARISONS.items():
    print(f"\n=== {name}: {base} vs {other} ===")
    CMP[name] = TASK_1 / "comparison" / name
    compare({base: EVAL[base], other: EVAL[other]}, out_dir=CMP[name])
    t = pd.read_csv(CMP[name] / "classwise_iou_test.csv")
    d = next(c for c in t if c.startswith("diff_"))
    print("Largest class-wise changes on test (positive = first model better):")
    display(t.reindex(t[d].abs().sort_values(ascending=False).index).head(6)[["class", "pixel_share_%", d]])

## Task 2 — RGB failure analysis (on the strongest RGB model, FCN 240×320, which Task 3 builds on)

In [ ]:
display(boundary_summary({k: EVAL[k] for k in ["DeepLabV3", "FCN", "FCN 240x320", "DeepLabV3 240x320"]}))
print("Most frequent confusions — FCN 240x320 (test)")
display(top_confusions(EVAL["FCN 240x320"], "test"))
print("Per-image mIoU by scene brightness and clutter (test)")
display(scene_breakdown({k: EVAL[k] for k in ["FCN", "FCN 240x320"]}))
worst = worst_images(EVAL["FCN 240x320"], "test", n=6)
display(worst)
show_examples({k: EVAL[k] for k in ["FCN", "FCN 240x320"]}, worst["id"].astype(str).tolist(), DATA_ROOT, "test",
              title="Worst FCN 240x320 test images")

## Task 3 — Depth analysis

Depth PNGs are 8-bit, min–max normalised **per image** (relative, not metric) and already hole-filled, so no missing-value
masking is needed; filled regions are least reliable on mirrors, windows and dark surfaces. Hypothesis from Task 2:
depth helps objects that **protrude** from their surroundings, not objects **coplanar** with what they're confused with.

In [ ]:
protruding = [("lamp", "wall"), ("refrigerator", "wall"), ("chair", "table"), ("person", "wall"), ("box", "otherprop")]
coplanar = [("whiteboard", "wall"), ("door", "wall"), ("mirror", "wall"), ("floor mat", "floor"), ("books", "bookshelf")]
for name, pairs in [("Protruding (depth expected to help)", protruding), ("Coplanar (depth expected NOT to help)", coplanar)]:
    print(name)
    display(confusion_change(EVAL["FCN 240x320"], EVAL["FCN 240x320 RGB-D mid"], pairs))
fused = ["FCN 240x320", "FCN 240x320 RGB-D mid", "FCN 240x320 RGB-D early"]
display(boundary_summary({k: EVAL[k] for k in fused}))
display(scene_breakdown({k: EVAL[k] for k in fused}))

In [ ]:
delta = per_image_delta(EVAL["FCN 240x320"], EVAL["FCN 240x320 RGB-D mid"], "test")
print("Mean per-image change:", round(delta.delta_mIoU_points.mean(), 2), "points;",
      f"{(delta.delta_mIoU_points > 0).mean():.0%} of test images improved")
runs = {k: EVAL[k] for k in ["FCN 240x320", "FCN 240x320 RGB-D mid"]}
show_examples(runs, delta.id.head(4).astype(str).tolist(), DATA_ROOT, "test", "Depth helped most", show_depth=True)
show_examples(runs, delta.id.tail(4).astype(str).tolist(), DATA_ROOT, "test", "Depth hurt most", show_depth=True)

In [ ]:
# Keep the analysis outputs (tables, figures) on Drive with the runs
if USE_DRIVE:
    for item in ["comparison", "master_results.csv", "learning_curves.png"]:
        src = TASK_1 / item
        dst = DRIVE_RUNS.parent / item
        (shutil.copytree(src, dst, dirs_exist_ok=True) if src.is_dir() else shutil.copy2(src, dst))
    print("Saved analysis outputs to", DRIVE_RUNS.parent)